# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


NAME(s) = "Ryan, Josh, Adalia"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

1. The Binary Classification task should use Binary Cross-Entropy Cost. The Multi-Class Classification should use Categorical Cross-Entropy Loss. The Regression task should use Mean Squared Error as the Cost Function.
2. Binary classification needs the output layer to be changed to use the sigmoid activation function to give probabilities for binary classification. Multiclass requires the output layer to use softmax. Regression does not require an activation function because the calculated values do not need to compressed between 0 and 1 or separated into discrete classes.
3. It is important to standardize the data before training each model so that the model's do not weigh features more heavily simply because they are larger. Standardizing the data ensures that all the inputs are equally considered.
4. Overfitting will show itself with extremely high training cost functions, but very low test cost functions. This displays that the model is overfitting on the training set and not becoming generalizable to other data.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [1]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [18]:
# importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [53]:
# Defining activation functions for use later
def sigmoid(z):
    return 1/(1+torch.exp(-z))

def relu(z):
    return torch.maximum(z, torch.tensor(0.0))

def tanh(z):
    return (torch.exp(z) - torch.exp(-z))/(torch.exp(z) + torch.exp(-z))

In [105]:
# 1. Implementation for One Hidden Layer
class BinaryMLP(object):
    def __init__(self, 
                 input_dim, # Think of j index above
                 hidden_dim,  # Think of k index above
                 act_func,
                 learning_rate=0.01):
        self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
        self.b_1 = torch.randn(1, hidden_dim, requires_grad=True)
        self.W_2 = torch.randn(hidden_dim, 1, requires_grad=True)
        self.b_2 = torch.randn(1, requires_grad=True)
        self.learning_rate = learning_rate
        self.act_func = act_func
        self.history = {"epoch": [], "cost": [], "acc": []}

    def forward(self, X):
        # 1-input to hidden units 
        h = torch.matmul(X, self.W_1) + self.b_1 
        # 2-Find output in each units through their activation function (ReLU in this case)
        h_act_func = self.act_func(h) 
        # 3- These outputs server as input to the last, output layer consisting of just one unit.
        out = torch.matmul(h_act_func, self.W_2) + self.b_2 
        return sigmoid(out)

    def cost(self, y_pred, y_true):
        # Clamp to avoid log(0)
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        log_cost = -torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
        return log_cost

    def fit(self, X, y, n_epochs=100):
        
        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii], y[ii]
                
                # Forward pass
                probs = self.forward(x_batch)
                
                # the cost function
                cost = self.cost(probs, y_batch)
            
                # Backward pass to calculate gradients
                cost.backward()
            
                # Gradient descent step (manual)
                with torch.no_grad():
                    self.W_1 -= self.learning_rate*self.W_1.grad
                    self.b_1 -= self.learning_rate*self.b_1.grad
                    self.W_2 -= self.learning_rate*self.W_2.grad
                    self.b_2 -= self.learning_rate*self.b_2.grad
                    # Update each weight
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_2.grad.zero_()
                    self.b_2.grad.zero_()
                    # Zero gradients 
        # evaluate each epoch while training and store results
            with torch.no_grad():
                final_epoch_cost = self.cost(self.forward(X), y).item()
            y_pred = self.predict(X)
            acc = (y_pred == y).float().mean().item()
            self.history["epoch"].append(epoch)
            self.history["cost"].append(final_epoch_cost)
            self.history["acc"].append(acc)
        return self
        
    def predict(self, X):
        with torch.no_grad():
            # Calculate probabilites of input data
            probs = self.forward(X)
            # Assign label takes all the predictions above 0.5 with boolean so becomes True then transforms 
            # True and False values into floats to become 0.0 and 1.0
            predicted = (probs>0.5).float() # COMES HERE (probs > 0.5 and convert to float)
        return predicted     

    def train_evaluation(self):
        return pd.DataFrame(self.history).set_index("epoch")

    def test_evaluation(self, X, y):
        with torch.no_grad():
            probs = self.forward(X)
            cost = self.cost(probs, y).item()
            y_pred = self.predict(X)
            acc = (y_pred == y).float().mean().item()
        print(f"Test Cost: {cost:.5f} and Test Accuracy {acc:.5f}")

In [7]:
# 1. Implementation for Two Hidden Layers
class BinaryMLP2(object):
    def __init__(self, 
                 input_dim, # Think of j index above
                 hidden_dim_1,  # Think of k index above
                 hidden_dim_2,
                 act_func,
                 learning_rate=0.01):
        self.W_1 = torch.randn(input_dim, hidden_dim_1, requires_grad=True)
        self.b_1 = torch.randn(1, hidden_dim_1, requires_grad=True)
        self.W_2 = torch.randn(hidden_dim_1, hidden_dim_2, requires_grad=True)
        self.b_2 = torch.randn(1, hidden_dim_2, requires_grad=True)
        self.W_3 = torch.randn(hidden_dim_2, 1, requires_grad=True)
        self.b_3 = torch.randn(1, requires_grad=True)
        self.learning_rate = learning_rate
        self.act_func = act_func
        self.history = {"epoch": [], "cost": [], "acc": []}

    def forward(self, X):
        h = torch.matmul(X, self.W_1) + self.b_1 
        h_act_func = self.act_func(h)         
        g = torch.matmul(h_act_func, self.W_2) + self.b_2 
        g_act_func = self.act_func(g)
        out = torch.matmul(g_act_func, self.W_3) + self.b_3
        return sigmoid(out)

    def cost(self, y_pred, y_true):
        # Clamp to avoid log(0)
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        log_cost = -torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
        return log_cost

    def fit(self, X, y, n_epochs=100):
        
        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii], y[ii]
                
                # Forward pass
                probs = self.forward(x_batch)
                
                # the cost function
                cost = self.cost(probs, y_batch)
            
                # Backward pass to calculate gradients
                cost.backward()
            
                # Gradient descent step (manual)
                with torch.no_grad():
                    self.W_1 -= self.learning_rate*self.W_1.grad
                    self.b_1 -= self.learning_rate*self.b_1.grad
                    self.W_2 -= self.learning_rate*self.W_2.grad
                    self.b_2 -= self.learning_rate*self.b_2.grad
                    self.W_3 -= self.learning_rate*self.W_3.grad
                    self.b_3 -= self.learning_rate*self.b_3.grad
                    # Update each weight
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_2.grad.zero_()
                    self.b_2.grad.zero_()
                    self.W_3.grad.zero_()
                    self.b_3.grad.zero_()
                    # Zero gradients 
        # evaluate each epoch while training and store results
            with torch.no_grad():
                final_epoch_cost = self.cost(self.forward(X), y).item()
            y_pred = self.predict(X)
            acc = (y_pred == y).float().mean().item()
            self.history["epoch"].append(epoch)
            self.history["cost"].append(final_epoch_cost)
            self.history["acc"].append(acc)
        return self
        
    def predict(self, X):
        with torch.no_grad():
            # Calculate probabilites of input data
            probs = self.forward(X)
            # Assign label takes all the predictions above 0.5 with boolean so becomes True then transforms 
            # True and False values into floats to become 0.0 and 1.0
            predicted = (probs>0.5).float() # COMES HERE (probs > 0.5 and convert to float)
        return predicted     

    def train_evaluation(self):
        return pd.DataFrame(self.history).set_index("epoch")

    def test_evaluation(self, X, y):
        with torch.no_grad():
            probs = self.forward(X)
            cost = self.cost(probs, y).item()
            y_pred = self.predict(X)
            acc = (y_pred == y).float().mean().item()
        print(f"Test Cost: {cost:.5f} and Test Accuracy {acc:.5f}")

In [84]:
# 2. Data Preparation

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Standardize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Numpy arrays to tensor for better flow
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train).float().reshape(-1,1)
y_test = torch.tensor(y_test).float().reshape(-1,1)

In [107]:
# 3. Model Training and Evaluation
# One hidden layer, varying the number of hidden units (use ReLU as their activation function)

# Create the model
mlp_model_6 = BinaryMLP(input_dim=X.shape[1], hidden_dim=6, act_func=relu)
# Fit the model
mlp_model_6.fit(X_train, y_train)
# Show training and testing results
train_df_6 = mlp_model_6.train_evaluation()
print(train_df_6)
mlp_model_6.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.336614  0.82000
1      0.299084  0.84375
2      0.290233  0.85250
3      0.286774  0.84750
4      0.284956  0.85125
...         ...      ...
95     0.065236  0.97375
96     0.064493  0.97500
97     0.065443  0.97375
98     0.064104  0.97500
99     0.063566  0.97500

[100 rows x 2 columns]
Test Cost: 0.10364 and Test Accuracy 0.96500


In [108]:
# Create the model
mlp_model_9 = BinaryMLP(input_dim=X.shape[1], hidden_dim=9, act_func=relu)
# Fit the model
mlp_model_9.fit(X_train, y_train)
# Show training and testing results for 9 hidden nodes
train_df_9 = mlp_model_9.train_evaluation()
print(train_df_9)
mlp_model_9.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.300676  0.87125
1      0.284613  0.87125
2      0.272130  0.88250
3      0.255683  0.89375
4      0.248296  0.89125
...         ...      ...
95     0.058283  0.97875
96     0.061342  0.97500
97     0.060432  0.97500
98     0.057752  0.98000
99     0.057359  0.98000

[100 rows x 2 columns]
Test Cost: 0.09070 and Test Accuracy 0.98000


In [109]:
# Create the model
mlp_model_12 = BinaryMLP(input_dim=X.shape[1], hidden_dim=12, act_func=relu)
# Fit the model
mlp_model_12.fit(X_train, y_train)
# Show training and testing results for 12 hidden nodes
train_df_12 = mlp_model_12.train_evaluation()
print(train_df_12)
mlp_model_12.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.291200  0.86875
1      0.266249  0.87625
2      0.254118  0.88625
3      0.238716  0.88750
4      0.226820  0.89875
...         ...      ...
95     0.055117  0.97875
96     0.054764  0.97875
97     0.054815  0.98000
98     0.054698  0.98000
99     0.054140  0.98000

[100 rows x 2 columns]
Test Cost: 0.08928 and Test Accuracy 0.98000


In [110]:
# 3. Model Training and Evaluation
# Same number of hidden units across one or two hidden layers (use ReLU as their activation function)

# Create the model
mlp_model_1hidden = BinaryMLP(input_dim=X.shape[1], hidden_dim=6, act_func=relu)
# Fit the model
mlp_model_1hidden.fit(X_train, y_train)
# Show training and testing results for 1 hidden layer
train_df_1hidden = mlp_model_1hidden.train_evaluation()
print(train_df_1hidden)
mlp_model_1hidden.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.278175  0.90125
1      0.247758  0.90125
2      0.229009  0.90625
3      0.213975  0.91250
4      0.197364  0.91625
...         ...      ...
95     0.058445  0.98000
96     0.058502  0.97500
97     0.060433  0.97500
98     0.057904  0.97500
99     0.058099  0.97750

[100 rows x 2 columns]
Test Cost: 0.08888 and Test Accuracy 0.98000


In [111]:
# Create the model
mlp_model_2hidden = BinaryMLP2(input_dim=X.shape[1], hidden_dim_1=6, hidden_dim_2=6, act_func=relu)
# Fit the model
mlp_model_2hidden.fit(X_train, y_train)
# Show training and testing results for 6 hidden nodes with 2 hidden layers
train_df_2hidden = mlp_model_2hidden.train_evaluation()
print(train_df_2hidden)
mlp_model_2hidden.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.309530  0.85500
1      0.286421  0.88125
2      0.262415  0.88750
3      0.257266  0.88750
4      0.250840  0.88750
...         ...      ...
95     0.059961  0.97375
96     0.056649  0.97500
97     0.063709  0.97500
98     0.078612  0.97125
99     0.065964  0.97125

[100 rows x 2 columns]
Test Cost: 0.12423 and Test Accuracy 0.94500


In [112]:
# 3. Model Training and Evaluation
# Repeat the above setups using Tanh activation instead of ReLU

# Create the model
mlp_model = BinaryMLP(input_dim=X.shape[1], hidden_dim=6, act_func=tanh)
# Fit the model
mlp_model.fit(X_train, y_train)
# Show training and testing results
train_df = mlp_model.train_evaluation()
print(train_df)
mlp_model.test_evaluation(X_test, y_test)

           cost      acc
epoch                   
0      0.288678  0.87375
1      0.256477  0.88625
2      0.229164  0.90375
3      0.208347  0.92875
4      0.192528  0.93125
...         ...      ...
95     0.068565  0.97500
96     0.068148  0.97625
97     0.067994  0.97750
98     0.068816  0.97375
99     0.067900  0.97500

[100 rows x 2 columns]
Test Cost: 0.09555 and Test Accuracy 0.98000


### 4. Reflection
- How does the number of hidden units affect performance?

Going from 6 to 9 to 12 hidden nodes improved cost for both train and test sets and improved accuracy.
  
- What changes when using two layers instead of one?

The neural network became less generalizable because it had lower accuracy and higher cost for the test set.

- How does the activation function (ReLU vs. Tanh) influence results?

The cost and accuracy metrics were both improved for training and testing.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [1]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [2]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
import torch
import torch.nn as nn

In [3]:
# 1. Implementation of One Hidden Layer
class MultiClassMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes, act_func=torch.relu, learning_rate=0.01):
        super().__init__()
        sizes = [input_dim] + hidden_dim + [n_classes]
        self.weights = nn.ParameterList()
        self.biases = nn.ParameterList()
        for i in range(len(sizes)-1):
            self.weights.append(nn.Parameter(torch.randn(sizes[i], sizes[i+1])))
            self.biases.append(nn.Parameter(torch.randn(1, sizes[i+1])))
        self.act_func = act_func
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.train_history = {"cost": [], "acc": []}
        self.history = {"model": [], "train_cost": [], "train_acc": [], "test_cost": [], "test_acc": []}
        
    def forward(self, X):
        for i in range(len(self.weights)-1):
            linear = torch.matmul(X, self.weights[i]) + self.biases[i]
            X = self.act_func(linear)
        out = torch.matmul(X, self.weights[-1]) + self.biases[-1]
        return torch.softmax(out, dim=1)

    def cost(self, y_pred, y_true):
        cost = 0
        for ii in range(y_true.shape[1]):
            y_oc = y_true[:, ii]
            y_pc = y_pred[:, ii]
            cost -= y_oc*torch.log(y_pc.clamp_min(1e-12))
        return cost.mean(0)

    def fit(self, X, y, n_epochs=100):
        self.train()
        for epoch in range(n_epochs):
            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii:ii+1], y[ii:ii+1]
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()
                with torch.no_grad():
                    for p in self.parameters():
                        p -= self.learning_rate * p.grad
                        p.grad.zero_() 
            with torch.no_grad():
                final_epoch_cost = self.cost(self.forward(X), y).item()
                probs = self.forward(X)
                y_pred = probs.argmax(dim=1)
                acc = (y_pred == y.argmax(dim=1)).float().mean().item()
            self.train_history["cost"].append(final_epoch_cost)
            self.train_history["acc"].append(acc)
        self.eval()
        return self

    def predict(self, X):
        self.eval()
        with torch.no_grad():
            return self.forward(X).argmax(dim=1)

    def test_evaluation(self, X, y):
        with torch.no_grad():
            probs = self.forward(X)
            test_cost = self.cost(probs, y).item()
            y_pred = probs.argmax(dim=1)
            test_acc = (y_pred == y.argmax(dim=1)).float().mean().item()
        self.history["test_cost"].append(test_cost)
        self.history["test_acc"].append(test_acc)
        self.history["train_cost"].append(self.train_history["cost"][-1])
        self.history["train_acc"].append(self.train_history["acc"][-1])
        self.history["model"].append(f"{len(self.hidden_dim)} layer, {self.hidden_dim[0]} units, {self.act_func.__name__}")
        return pd.DataFrame(self.history)

In [4]:
# 2. Data Preparation

# One Hot Encode before split
y_re = y.reshape(-1,1)
onehot = OneHotEncoder(sparse_output=False)
y = onehot.fit_transform(y_re)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Numpy arrays to tensor for better flow
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train).float()
y_test = torch.tensor(y_test).float()

In [5]:
# 3. Model Training and Evaluation
# One hidden layer, varying the number of hidden units (use ReLU as their activation function)

torch.manual_seed(0)
# Create the model
multi_mlp_model_6 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[6], n_classes=3)
# Fit the model
multi_mlp_model_6.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x6]
      (1): Parameter containing: [torch.float32 of size 6x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x6]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [6]:
# 3. Model Training and Evaluation
# One hidden layer, varying the number of hidden units (use ReLU as their activation function)
torch.manual_seed(0)
# Create the model
multi_mlp_model_9 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[9], n_classes=3)
# Fit the model
multi_mlp_model_9.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x9]
      (1): Parameter containing: [torch.float32 of size 9x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x9]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [7]:
# 3. Model Training and Evaluation
# One hidden layer, varying the number of hidden units (use ReLU as their activation function)
torch.manual_seed(0)
# Create the model
multi_mlp_model_12 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[12], n_classes=3)
# Fit the model
multi_mlp_model_12.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x12]
      (1): Parameter containing: [torch.float32 of size 12x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x12]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [8]:
# 3. Model Training and Evaluation
# Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
torch.manual_seed(0)
# Create the model
multi_mlp_model_2hidden = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[12,12], n_classes=3)
# Fit the model
multi_mlp_model_2hidden.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x12]
      (1): Parameter containing: [torch.float32 of size 12x12]
      (2): Parameter containing: [torch.float32 of size 12x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x12]
      (1): Parameter containing: [torch.float32 of size 1x12]
      (2): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [9]:
# 3. Model Training and Evaluation
# Repeat ALL of the above setups using Tanh activation instead of ReLU
torch.manual_seed(0)
# Create the model
multi_mlp_model_tanh1 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[6], act_func=torch.tanh, n_classes=3)
# Fit the model
multi_mlp_model_tanh1.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x6]
      (1): Parameter containing: [torch.float32 of size 6x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x6]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [10]:
# 3. Model Training and Evaluation
# Repeat ALL of the above setups using Tanh activation instead of ReLU
torch.manual_seed(0)
# Create the model
multi_mlp_model_tanh2 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[9], act_func=torch.tanh, n_classes=3)
# Fit the model
multi_mlp_model_tanh2.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x9]
      (1): Parameter containing: [torch.float32 of size 9x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x9]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [11]:
# 3. Model Training and Evaluation
# One hidden layer, varying the number of hidden units (use ReLU as their activation function)
torch.manual_seed(0)
# Create the model
multi_mlp_model_tanh3 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[12], act_func=torch.tanh, n_classes=3)
# Fit the model
multi_mlp_model_tanh3.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x12]
      (1): Parameter containing: [torch.float32 of size 12x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x12]
      (1): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [12]:
# 3. Model Training and Evaluation
# Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
torch.manual_seed(0)
# Create the model
multi_mlp_model_tanh4 = MultiClassMLP(input_dim=X.shape[1], hidden_dim=[12,12], act_func=torch.tanh, n_classes=3)
# Fit the model
multi_mlp_model_tanh4.fit(X_train, y_train)

MultiClassMLP(
  (weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 2x12]
      (1): Parameter containing: [torch.float32 of size 12x12]
      (2): Parameter containing: [torch.float32 of size 12x3]
  )
  (biases): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x12]
      (1): Parameter containing: [torch.float32 of size 1x12]
      (2): Parameter containing: [torch.float32 of size 1x3]
  )
)

In [13]:
df1 = multi_mlp_model_6.test_evaluation(X_test, y_test)
df2 = multi_mlp_model_9.test_evaluation(X_test, y_test)
df3 = multi_mlp_model_12.test_evaluation(X_test, y_test)
df4 = multi_mlp_model_2hidden.test_evaluation(X_test, y_test)
df5 = multi_mlp_model_tanh1.test_evaluation(X_test, y_test)
df6 = multi_mlp_model_tanh2.test_evaluation(X_test, y_test)
df7 = multi_mlp_model_tanh3.test_evaluation(X_test, y_test)
df8 = multi_mlp_model_tanh4.test_evaluation(X_test, y_test)

In [14]:
df_list = [df1,df2,df3,df4,df5,df6,df7,df8]
combined_results = pd.concat(df_list, axis=0, ignore_index=True)

In [15]:
combined_results

,model,train_cost,train_acc,test_cost,test_acc
0,"1 layer, 6 units, relu",0.553320,0.82125,0.621847,0.795
1,"1 layer, 9 units, relu",0.560691,0.80750,0.624367,0.795
2,"1 layer, 12 units, relu",0.550080,0.81375,0.598650,0.795
3,"2 layer, 12 units, relu",0.522972,0.82375,0.577198,0.815
4,"1 layer, 6 units, tanh",0.540266,0.82000,0.576760,0.800
5,"1 layer, 9 units, tanh",0.551444,0.80625,0.614633,0.790
6,"1 layer, 12 units, tanh",0.543770,0.81750,0.595210,0.800
7,"2 layer, 12 units, tanh",0.508893,0.82875,0.554782,0.815


### 4. Reflection
- How does the number of hidden units affect performance?

The number of hidden units does not cause a significant change to performance with training cost and accuracy hovering around 0.55 and 0.81 respectively. Test cost and accuracy lies around 0.6 and 0.795 respectively. The main issue is that this problem is not complex enough that adding more units makes a true difference in performance.

- What changes when using two layers instead of one?

The metrics did improve across the board, however the changes themselves were minimal. Training cost and accuracy went from 0.55 and 0.81 to 0.52 and 0.82 respectively. Test cost and accuracy went from 0.60 and 0.795 to 0.58 and 0.815. The same issue remains here though, because a 2 layer neural network did not result in a large performance improvement because of the simple classification problem.

- How does the activation function (ReLU vs. Tanh) influence results?

Generally, using the Tanh activation function did slightly improve results, but the changes were very small. Usually about 0.1-0.2 different for both cost and accuracy metrics.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE